In [1]:
import os, sys, subprocess

print("Python:", sys.version)
subprocess.run(["nvidia-smi"])

import torch
print("\nTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tue Jul 28 13:33:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |             

In [2]:
DATA_ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"

def tree(path, prefix="", max_depth=3, depth=0, max_items=20):
    if depth > max_depth:
        return
    try:
        items = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    for i, item in enumerate(items[:max_items]):
        full = os.path.join(path, item)
        print(prefix + ("📁 " if os.path.isdir(full) else "📄 ") + item)
        if os.path.isdir(full):
            tree(full, prefix + "  ", max_depth, depth + 1, max_items)
    if len(items) > max_items:
        print(prefix + f"... and {len(items) - max_items} more")

print(f"Root exists: {os.path.exists(DATA_ROOT)}")
tree(DATA_ROOT, max_depth=3)

Root exists: True
📁 dataset_kaggle
  📁 annotations
    📄 test_annotations.json
    📄 train_annotations.json
  📁 extracted_frames
    📁 test
      📁 TEST_IF_001
      📁 TEST_IF_002
      📁 TEST_IF_003
      📁 TEST_IF_004
      📁 TEST_IF_005
      📁 TEST_IF_006
      📁 TEST_IF_007
      📁 TEST_IF_008
      📁 TEST_IF_009
      📁 TEST_IF_010
      📁 TEST_IF_011
      📁 TEST_IF_012
      📁 TEST_IF_013
      📁 TEST_IF_014
      📁 TEST_IF_015
      📁 TEST_IF_016
      📁 TEST_IF_017
      📁 TEST_IF_018
      📁 TEST_IF_019
      📁 TEST_IF_020
      ... and 780 more
    📁 train
      📁 IF_001
      📁 IF_002
      📁 IF_003
      📁 IF_004
      📁 IF_005
      📁 IF_006
      📁 IF_007
      📁 IF_008
      📁 IF_009
      📁 IF_010
      📁 IF_011
      📁 IF_012
      📁 IF_013
      📁 IF_014
      📁 IF_015
      📁 IF_016
      📁 IF_017
      📁 IF_018
      📁 IF_019
      📁 IF_020
      ... and 1580 more
  📁 extracted_text
    📄 test_metadata.csv
    📄 test_vision_text.jsonl
    📄 train_metadata.csv
    

In [3]:
import pandas as pd

csv_files = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".csv", ".json", ".jsonl")):
            csv_files.append(os.path.join(root, f))

print(f"Found {len(csv_files)} metadata files:")
for f in csv_files:
    print(" -", f)

# Inspect each CSV found
for f in csv_files:
    if f.endswith(".csv"):
        print(f"\n{'='*80}\n{f}\n{'='*80}")
        try:
            df = pd.read_csv(f)
            print("Shape:", df.shape)
            print("Columns:", list(df.columns))
            print(df.head(3))
        except Exception as e:
            print("Error reading:", e)

Found 6 metadata files:
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/annotations/test_annotations.json
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/annotations/train_annotations.json
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_vision_text.jsonl
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/test_vision_text.jsonl
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_metadata.csv
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/test_metadata.csv

/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_metadata.csv
Shape: (1600, 6)
Columns: ['video_id', 'has_audio', 'transcript', 'ocr_text', 'category', 'subcategory']
  v

In [5]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "accelerate", "torchaudio", "soundfile", "librosa"])

import os, json, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"
DS = os.path.join(ROOT, "dataset_kaggle")
FRAMES_DIR = os.path.join(DS, "extracted_frames")
TEXT_DIR = os.path.join(DS, "extracted_text")
AUDIO_DIR = os.path.join(ROOT, "extracted_audio", "extracted_audio")

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

Using device: cuda:0


In [6]:
def load_split(split):
    csv_path = os.path.join(TEXT_DIR, f"{split}_metadata.csv")
    df = pd.read_csv(csv_path)
    df["split"] = split
    return df

train_df = load_split("train")
test_df = load_split("test")

print("train:", train_df.shape, "test:", test_df.shape)
print(train_df["category"].value_counts())
print(test_df["category"].value_counts())

# Binary label: safe=0, misleading=1
def to_binary(cat):
    c = str(cat).strip().lower()
    if c == "safe":
        return 0
    elif c == "misleading":
        return 1
    else:
        raise ValueError(f"Unexpected category value: {cat}")

train_df["label"] = train_df["category"].apply(to_binary)
test_df["label"] = test_df["category"].apply(to_binary)

def frame_dir_for(row):
    return os.path.join(FRAMES_DIR, row["split"], row["video_id"])

def audio_path_for(row):
    return os.path.join(AUDIO_DIR, row["split"], row["video_id"] + ".wav")

for df in (train_df, test_df):
    df["frame_dir"] = df.apply(frame_dir_for, axis=1)
    df["audio_path"] = df.apply(audio_path_for, axis=1)
    df["frame_dir_exists"] = df["frame_dir"].apply(os.path.isdir)
    df["audio_exists"] = df["audio_path"].apply(os.path.isfile)

print("\nMissing frame dirs (train):", (~train_df["frame_dir_exists"]).sum())
print("Missing audio files (train):", (~train_df["audio_exists"]).sum())
print("Missing frame dirs (test):", (~test_df["frame_dir_exists"]).sum())
print("Missing audio files (test):", (~test_df["audio_exists"]).sum())

# Peek inside one frame folder to confirm naming pattern
sample_dir = train_df.iloc[0]["frame_dir"]
print("\nSample frame dir:", sample_dir)
print(sorted(os.listdir(sample_dir))[:5])

train: (1600, 7) test: (800, 7)
category
misleading    800
safe          800
Name: count, dtype: int64
category
misleading    400
safe          400
Name: count, dtype: int64

Missing frame dirs (train): 0
Missing audio files (train): 0
Missing frame dirs (test): 0
Missing audio files (test): 0

Sample frame dir: /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_frames/train/IF_001
['frame_01.jpg', 'frame_02.jpg', 'frame_03.jpg', 'frame_04.jpg', 'frame_05.jpg']


In [7]:
# Diagnostic — run this whole cell and paste ALL output
for name, df in [("train", train_df), ("test", test_df)]:
    print(f"=== {name} ===")
    print("  Missing frame dirs :", (~df["frame_dir_exists"]).sum(), "/", len(df))
    print("  Missing audio files:", (~df["audio_exists"]).sum(), "/", len(df))
    print("  label counts:", df["label"].value_counts().to_dict())

# Sample frame folder contents
sample_dir = train_df.iloc[0]["frame_dir"]
print("\nSample frame dir:", sample_dir)
files = sorted(os.listdir(sample_dir))
print("  num files:", len(files))
print("  first 8   :", files[:8])

# Frame count distribution across a handful of videos
import numpy as np
counts = [len(os.listdir(d)) for d in train_df["frame_dir"].head(50) if os.path.isdir(d)]
print("\nFrame counts (first 50 train vids): min", np.min(counts), "max", np.max(counts), "median", int(np.median(counts)))

# Confirm a JSONL text row structure (in case we want vision_text later)
jsonl_path = os.path.join(TEXT_DIR, "train_vision_text.jsonl")
with open(jsonl_path) as f:
    first = f.readline().strip()
print("\nvision_text.jsonl first row (truncated):", first[:300])

=== train ===
  Missing frame dirs : 0 / 1600
  Missing audio files: 0 / 1600
  label counts: {1: 800, 0: 800}
=== test ===
  Missing frame dirs : 0 / 800
  Missing audio files: 0 / 800
  label counts: {1: 400, 0: 400}

Sample frame dir: /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_frames/train/IF_001
  num files: 16
  first 8   : ['frame_01.jpg', 'frame_02.jpg', 'frame_03.jpg', 'frame_04.jpg', 'frame_05.jpg', 'frame_06.jpg', 'frame_07.jpg', 'frame_08.jpg']

Frame counts (first 50 train vids): min 16 max 16 median 16

vision_text.jsonl first row (truncated): {"video_id": "IF_001", "vision_text_all": ["OR EVEN TAKA ON YOUR FIRST", "CRAZY TIME"], "vision_text_topk": ["OR EVEN TAKA ON YOUR FIRST", "CRAZY TIME"], "support": [{"text": "OR EVEN TAKA ON YOUR FIRST", "frames": 2}, {"text": "CRAZY TIME", "frames": 2}]}


In [8]:
from transformers import (
    CLIPModel, CLIPProcessor,
    Wav2Vec2Model, Wav2Vec2FeatureExtractor,
    AutoModel, AutoTokenizer,
)
import torchaudio, librosa

print("Loading CLIP ViT-B/32 ...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Loading Wav2Vec2 ...")
w2v_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE).eval()
w2v_fe = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")

print("Loading XLM-RoBERTa ...")
xlmr_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr_model = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE).eval()

# Qwen embedding model — small, fits T4. If download is slow, this is the one heavy download.
print("Loading Qwen embedding model ...")
QWEN_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
try:
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)
    qwen_model = AutoModel.from_pretrained(QWEN_NAME, trust_remote_code=True,
                                           torch_dtype=torch.float16).to(DEVICE).eval()
    QWEN_OK = True
    print("Qwen loaded.")
except Exception as e:
    print("Qwen load failed, will fall back to a second XLM-R pooling as 'Qwen' slot:", e)
    QWEN_OK = False

for m in [clip_model, w2v_model, xlmr_model]:
    for p in m.parameters():
        p.requires_grad = False
print("Encoders ready.")

Loading CLIP ViT-B/32 ...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/901 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Qwen load failed, will fall back to a second XLM-R pooling as 'Qwen' slot: No module named 'transformers.models.qwen2.tokenization_qwen2_fast'
Encoders ready.


In [9]:
# The gte-Qwen model ships custom tokenizer code that breaks on some transformers
# versions. Force the standard Qwen2 tokenizer + slow tokenizer to bypass it.
from transformers import AutoModel, AutoTokenizer

QWEN_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
QWEN_OK = False

try:
    qwen_tok = AutoTokenizer.from_pretrained(
        QWEN_NAME, trust_remote_code=False, use_fast=False
    )
    qwen_model = AutoModel.from_pretrained(
        QWEN_NAME, trust_remote_code=True, torch_dtype=torch.float16
    ).to(DEVICE).eval()
    for p in qwen_model.parameters():
        p.requires_grad = False
    QWEN_OK = True
    print("✅ Qwen loaded via standard tokenizer (Option A).")
except Exception as e:
    print("Option A failed:", repr(e)[:200])

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

...

modeling_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct:
- modeling_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Option A failed: AttributeError("'Qwen2Config' object has no attribute 'rope_theta'")


In [10]:
QWEN_NAME = "Qwen/Qwen3-Embedding-0.6B"
QWEN_OK = False
try:
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME)
    qwen_model = AutoModel.from_pretrained(QWEN_NAME, dtype=torch.float16).to(DEVICE).eval()
    for p in qwen_model.parameters():
        p.requires_grad = False
    QWEN_OK = True
    print("✅ Qwen loaded (Option B: Qwen3-Embedding-0.6B).")
except Exception as e:
    print("Option B failed:", repr(e)[:300])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ Qwen loaded (Option B: Qwen3-Embedding-0.6B).


In [11]:
if QWEN_OK:
    with torch.no_grad():
        _p = qwen_tok("test", return_tensors="pt", truncation=True,
                      max_length=16, padding="max_length").to(DEVICE)
        _o = qwen_model(**_p).last_hidden_state
        QWEN_DIM = _o.shape[-1]
    print("QWEN_OK =", QWEN_OK, "| QWEN_DIM =", QWEN_DIM)
else:
    QWEN_DIM = XLMR_DIM
    print("Qwen unavailable — XLM-R only.")

QWEN_OK = True | QWEN_DIM = 1024


In [15]:
paths = sorted(glob.glob(os.path.join(train_df.iloc[0]["frame_dir"], "*.jpg")))
imgs = [Image.open(p).convert("RGB") for p in paths]
inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    out = clip_model.get_image_features(**inp)

print("type:", type(out))
print("is tensor:", torch.is_tensor(out))
if hasattr(out, "keys"):
    print("keys:", list(out.keys()))
print("attrs:", [a for a in dir(out) if not a.startswith("_")][:30])

type: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
is tensor: False
keys: ['last_hidden_state', 'pooler_output']
attrs: ['attentions', 'clear', 'copy', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'last_hidden_state', 'move_to_end', 'pooler_output', 'pop', 'popitem', 'setdefault', 'to_tuple', 'update', 'values']


In [16]:
CLIP_DIM = 512   # CLIP joint-space image embedding dim (after visual_projection)
W2V_DIM  = 768
XLMR_DIM = 768

@torch.no_grad()
def extract_visual(frame_dir):
    """True CLIP image embeddings (16, 512) = vision_model pooled -> visual_projection."""
    paths = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))
    imgs = [Image.open(p).convert("RGB") for p in paths]
    inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)
    vision_out = clip_model.vision_model(pixel_values=inp["pixel_values"])
    pooled = vision_out.pooler_output                 # (16, 768)
    feats = clip_model.visual_projection(pooled)      # (16, 512)
    feats = F.normalize(feats, dim=-1)
    return feats.cpu().float().numpy()

@torch.no_grad()
def extract_audio(audio_path):
    try:
        wav, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return np.zeros(W2V_DIM, dtype=np.float32)
    if wav.size == 0:
        return np.zeros(W2V_DIM, dtype=np.float32)
    wav = wav[: 16000 * 20]
    inp = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    out = w2v_model(**inp).last_hidden_state
    return out.mean(dim=1).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def _mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def extract_text_xlmr(text):
    inp = xlmr_tok(text, return_tensors="pt", truncation=True,
                   max_length=128, padding="max_length").to(DEVICE)
    out = xlmr_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def extract_text_qwen(text):
    inp = qwen_tok(text, return_tensors="pt", truncation=True,
                   max_length=128, padding="max_length").to(DEVICE)
    out = qwen_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

def build_text(row):
    t = "" if pd.isna(row["transcript"]) else str(row["transcript"])
    o = "" if pd.isna(row["ocr_text"]) else str(row["ocr_text"])
    combined = (t + " " + o).strip().lower()
    return combined if combined else "no text"

# smoke test
_v = extract_visual(train_df.iloc[0]["frame_dir"])
_a = extract_audio(train_df.iloc[0]["audio_path"])
_tx = extract_text_xlmr(build_text(train_df.iloc[0]))
_tq = extract_text_qwen(build_text(train_df.iloc[0]))
print("smoke test -> vis:", _v.shape, "aud:", _a.shape, "xlmr:", _tx.shape, "qwen:", _tq.shape)

smoke test -> vis: (16, 512) aud: (768,) xlmr: (768,) qwen: (1024,)


In [19]:
with torch.no_grad():
    _p = qwen_tok("test", return_tensors="pt", truncation=True,
                  max_length=16, padding="max_length").to(DEVICE)
    QWEN_DIM = qwen_model(**_p).last_hidden_state.shape[-1]
print("QWEN_DIM is now:", QWEN_DIM)

QWEN_DIM is now: 1024


In [20]:
import gc

def extract_split(df, split):
    N = len(df)
    vis = np.zeros((N, 16, CLIP_DIM), dtype=np.float32)
    aud = np.zeros((N, W2V_DIM), dtype=np.float32)
    txl = np.zeros((N, XLMR_DIM), dtype=np.float32)
    tqw = np.zeros((N, QWEN_DIM), dtype=np.float32)   # QWEN_DIM = 1024 now
    lab = df["label"].values.astype(np.int64)
    vids = df["video_id"].tolist()

    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=N, desc=f"extract {split}")):
        vis[i] = extract_visual(row["frame_dir"])
        aud[i] = extract_audio(row["audio_path"])
        text = build_text(row)
        txl[i] = extract_text_xlmr(text)
        tqw[i] = extract_text_qwen(text)
        if (i + 1) % 100 == 0:
            torch.cuda.empty_cache(); gc.collect()

    out = os.path.join(WORK, f"feats_{split}.npz")
    np.savez_compressed(out, vis=vis, aud=aud, txl=txl, tqw=tqw, lab=lab,
                        vids=np.array(vids))
    print("saved", out, "| shapes:", vis.shape, aud.shape, txl.shape, tqw.shape)
    return out

# sanity check before the long run
print("QWEN_DIM in use:", QWEN_DIM)
assert QWEN_DIM == 1024, "QWEN_DIM mismatch — re-run the probe cell"

extract_split(train_df, "train")
extract_split(test_df, "test")
print("PHASE A complete.")

QWEN_DIM in use: 1024


extract train:   0%|          | 0/1600 [00:00<?, ?it/s]

saved /kaggle/working/feats_train.npz | shapes: (1600, 16, 512) (1600, 768) (1600, 768) (1600, 1024)


extract test:   0%|          | 0/800 [00:00<?, ?it/s]

saved /kaggle/working/feats_test.npz | shapes: (800, 16, 512) (800, 768) (800, 768) (800, 1024)
PHASE A complete.


In [21]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix)
from sklearn.model_selection import StratifiedKFold
import pandas as pd

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

tr = np.load("/kaggle/working/feats_train.npz", allow_pickle=True)
te = np.load("/kaggle/working/feats_test.npz", allow_pickle=True)

# frame mean-pool for visual (temporal attention added inside the model variants)
def pack(split):
    return {
        "vis_seq": torch.tensor(split["vis"]),           # (N,16,512)
        "vis": torch.tensor(split["vis"].mean(1)),       # (N,512) simple mean (for baselines)
        "aud": torch.tensor(split["aud"]),               # (N,768)
        "txl": torch.tensor(split["txl"]),               # (N,768)
        "tqw": torch.tensor(split["tqw"]),               # (N,1024)
        "lab": torch.tensor(split["lab"]).long(),        # (N,)
    }
TR, TE = pack(tr), pack(te)
DIMS = {"vis":512, "aud":768, "txl":768, "tqw":1024}
print("loaded. train:", TR["lab"].shape[0], "test:", TE["lab"].shape[0])
print("class balance train:", torch.bincount(TR["lab"]).tolist(),
      "| test:", torch.bincount(TE["lab"]).tolist())

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred)*100,
        "Precision": precision_score(y_true, y_pred, zero_division=0)*100,
        "Recall":    recall_score(y_true, y_pred, zero_division=0)*100,
        "Macro F1":  f1_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)*100,
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "MCC":       matthews_corrcoef(y_true, y_pred),
    }

def fmt(d):
    return {k: (round(v,2) if k not in ("ROC-AUC","MCC") else round(v,3)) for k,v in d.items()}

loaded. train: 1600 test: 800
class balance train: [800, 800] | test: [400, 400]


In [23]:
class ModalityProj(nn.Module):
    def __init__(self, in_dim, out_dim=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim),
                                 nn.GELU(), nn.Dropout(p))
    def forward(self, x):
        return self.net(x)

class TemporalAttn(nn.Module):
    """Attention pool over 16 frame embeddings -> single visual vector."""
    def __init__(self, dim=512):
        super().__init__()
        self.w = nn.Linear(dim, 1)
    def forward(self, seq):                                  # (B,16,512)
        a = torch.softmax(self.w(seq).squeeze(-1), dim=1)    # (B,16)
        return (a.unsqueeze(-1) * seq).sum(1)                # (B,512)

class CrossAttn(nn.Module):
    def __init__(self, dim=256, heads=4):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
    def forward(self, q, kv):
        q, kv = q.unsqueeze(1), kv.unsqueeze(1)
        o, _ = self.mha(q, kv, kv)
        return o.squeeze(1)

class FusionModel(nn.Module):
    """
    mode: 'concat' | 'gated' | 'attn' | 'cross' | 'proposed' (CA-GF-AMW)
    mods: subset of ['vis','aud','txl','tqw']
    use_temporal: attention-pool frames instead of mean
    """
    def __init__(self, mods, mode="proposed", d=256, use_temporal=True):
        super().__init__()
        self.mods, self.mode, self.d = mods, mode, d
        self.use_temporal = use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d) for m in mods})
        M = len(mods)
        if mode in ("cross", "proposed") and M >= 2:
            self.cross = CrossAttn(d)
        if mode in ("gated", "proposed"):
            self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        if mode in ("attn", "proposed"):
            self.wgt = nn.Linear(d*M, M)
        fused_dim = d*M if mode in ("concat", "gated") else d
        self.head = nn.Sequential(nn.Linear(fused_dim, 128), nn.LayerNorm(128),
                                  nn.GELU(), nn.Dropout(0.3), nn.Linear(128, 2))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            elif m == "vis":
                feats[m] = self.proj[m](batch["vis"])
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if self.mode == "concat":
            fused = cat
        elif self.mode == "gated":
            fused = self.gate(cat) * cat
        elif self.mode == "attn":
            w = torch.softmax(self.wgt(cat), dim=-1)
            fused = sum(w[:, i:i+1] * mats[i] for i in range(M))
        elif self.mode == "cross":
            if M >= 2:
                fused = self.cross(mats[0], mats[1]) + sum(mats) / M
            else:
                fused = mats[0]
        elif self.mode == "proposed":
            if M >= 2:
                others = sum(mats[1:]) / max(M - 1, 1)
                fcross = self.cross(mats[0], others) + mats[0]
            else:
                fcross = mats[0]
            fgated = self.gate(cat) * cat
            fgated = fgated.view(cat.size(0), M, self.d).mean(1)
            w = torch.softmax(self.wgt(cat), dim=-1)
            fadapt = sum(w[:, i:i+1] * mats[i] for i in range(M))
            fused = fcross + fgated + fadapt
        return self.head(fused)

def move(batch, idx=None):
    out = {}
    for k, v in batch.items():
        out[k] = (v[idx] if idx is not None else v).to(DEVICE)
    return out

print("B2 defined OK.")

B2 defined OK.


In [24]:
def train_eval(mods, mode="proposed", use_temporal=True,
               train=TR, test=TE, epochs=30, lr=1e-3, bs=64, verbose=False, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModel(mods, mode, use_temporal=use_temporal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    lossf = nn.CrossEntropyLoss()
    N = train["lab"].shape[0]
    idx_all = np.arange(N)

    for ep in range(epochs):
        model.train(); np.random.shuffle(idx_all)
        for s in range(0, N, bs):
            bidx = idx_all[s:s+bs]
            b = move(train, bidx)
            logits = model(b)
            loss = lossf(logits, b["lab"])
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        b = move(test)
        logits = model(b)
        prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
        true = test["lab"].numpy()
    return compute_metrics(true, pred, prob), model

print("B3 defined OK.")

B3 defined OK.


In [25]:
metrics, _ = train_eval(["vis","aud","txl","tqw"], mode="proposed",
                        use_temporal=True, epochs=40, lr=1e-3)
print("PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW):")
for k,v in fmt(metrics).items():
    print(f"  {k:12s}: {v}")

PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW):
  Accuracy    : 87.25
  Precision   : 86.7
  Recall      : 88.0
  Macro F1    : 87.25
  Weighted F1 : 87.25
  ROC-AUC     : 0.936
  MCC         : 0.745


In [26]:
print("=== UNIMODAL (single modality, proposed head) ===\n")
uni = {"CLIP (visual)":["vis"], "Wav2Vec2 (audio)":["aud"],
       "XLM-R (text)":["txl"], "Qwen (text)":["tqw"]}
rows=[]
for name, mods in uni.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Model"]=name; rows.append(r)
df_uni = pd.DataFrame(rows)[["Model","Accuracy","Macro F1","Weighted F1","ROC-AUC","MCC"]]
print(df_uni.to_string(index=False))

=== UNIMODAL (single modality, proposed head) ===

           Model  Accuracy  Macro F1  Weighted F1  ROC-AUC   MCC
   CLIP (visual)     77.25     77.25        77.25    0.854 0.545
Wav2Vec2 (audio)     62.88     62.82        62.82    0.643 0.258
    XLM-R (text)     84.50     84.50        84.50    0.929 0.690
     Qwen (text)     86.88     86.86        86.86    0.912 0.740


In [27]:
print("=== BIMODAL & TRIMODAL ===\n")
combos = {
    "CLIP + Wav2Vec2":              ["vis","aud"],
    "CLIP + XLM-R":                 ["vis","txl"],
    "CLIP + Qwen":                  ["vis","tqw"],
    "CLIP + XLM-R + Qwen":          ["vis","txl","tqw"],
    "Wav2Vec2 + XLM-R + Qwen":      ["aud","txl","tqw"],
    "CLIP + Wav2Vec2 + XLM-R":      ["vis","aud","txl"],
    "CLIP + Wav2Vec2 + Qwen":       ["vis","aud","tqw"],
    "CLIP+Wav2Vec2+XLM-R+Qwen":     ["vis","aud","txl","tqw"],
}
rows=[]
for name, mods in combos.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Model"]=name; rows.append(r)
df_multi = pd.DataFrame(rows)[["Model","Accuracy","Macro F1","ROC-AUC","MCC"]]
print(df_multi.to_string(index=False))

=== BIMODAL & TRIMODAL ===

                   Model  Accuracy  Macro F1  ROC-AUC   MCC
         CLIP + Wav2Vec2     78.00     77.93    0.871 0.564
            CLIP + XLM-R     84.50     84.49    0.932 0.691
             CLIP + Qwen     86.25     86.25    0.937 0.725
     CLIP + XLM-R + Qwen     86.38     86.37    0.940 0.728
 Wav2Vec2 + XLM-R + Qwen     86.75     86.75    0.926 0.735
 CLIP + Wav2Vec2 + XLM-R     85.25     85.25    0.925 0.705
  CLIP + Wav2Vec2 + Qwen     85.38     85.36    0.937 0.708
CLIP+Wav2Vec2+XLM-R+Qwen     87.25     87.25    0.936 0.745


In [28]:
print("=== FUSION STRATEGIES (all 4 modalities) ===\n")
strategies = {"Concatenation":"concat", "Gated Fusion":"gated",
              "Attention-Based":"attn", "Cross-Attention":"cross",
              "Proposed (CA-GF-AMW)":"proposed"}
rows=[]
for name, mode in strategies.items():
    m,_ = train_eval(["vis","aud","txl","tqw"], mode=mode, use_temporal=True, epochs=40)
    r = fmt(m); r["Fusion"]=name; rows.append(r)
df_fus = pd.DataFrame(rows)[["Fusion","Accuracy","Macro F1","ROC-AUC","MCC"]]
print(df_fus.to_string(index=False))

=== FUSION STRATEGIES (all 4 modalities) ===

              Fusion  Accuracy  Macro F1  ROC-AUC   MCC
       Concatenation     85.38     85.37    0.939 0.708
        Gated Fusion     85.62     85.62    0.941 0.713
     Attention-Based     85.62     85.62    0.934 0.713
     Cross-Attention     85.38     85.35    0.947 0.710
Proposed (CA-GF-AMW)     87.25     87.25    0.936 0.745


In [29]:
print("=== ABLATION (remove one component) ===\n")
abl = {
    "Full Model":            ["vis","aud","txl","tqw"],
    "Without Audio":         ["vis","txl","tqw"],
    "Without Vision":        ["aud","txl","tqw"],
    "Without Text":          ["vis","aud"],
}
rows=[]
for name, mods in abl.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Configuration"]=name; rows.append(r)
# without fusion = concat baseline on all modalities
m,_ = train_eval(["vis","aud","txl","tqw"], mode="concat", use_temporal=True, epochs=40)
r = fmt(m); r["Configuration"]="Without Fusion Module"; rows.append(r)
df_abl = pd.DataFrame(rows)[["Configuration","Accuracy","Macro F1","MCC"]]
print(df_abl.to_string(index=False))

=== ABLATION (remove one component) ===

        Configuration  Accuracy  Macro F1   MCC
           Full Model     87.25     87.25 0.745
        Without Audio     86.38     86.37 0.728
       Without Vision     86.75     86.75 0.735
         Without Text     78.00     77.93 0.564
Without Fusion Module     85.38     85.37 0.708
